# Advanced Python Functions & Descriptors — 22 Worked Problems

**Topic derived from the supplied lesson:** Python functions implement the *non-data descriptor* protocol; reading a function through an instance creates a bound method; `__get__`, `__func__`, `__self__`, and `types.MethodType` explain how the binding works.

**What this notebook adds:** 22 increasingly advanced problems with runnable solutions, numerous examples, defensive tests, descriptor-precedence experiments, inheritance, decorator order, caching, tracing, and a realistic capstone. All examples use only the Python standard library (Python 3.10+). Run **Kernel → Restart & Run All** from top to bottom.

**How to study:** For every problem, read the challenge, try a scratch cell before looking at the immediately following solution, and run its assertions. Problem names and local classes are intentionally distinct to keep the notebook safe to run sequentially. Assertions verify behavior; comments explain *why* it occurs, rather than relying on version-dependent memory addresses or method representations.

**Source boundary:** The supplied lesson directly teaches basic function `__get__` binding, the method object's `__func__`, and using `types.MethodType` in a homemade descriptor. Later topics are **additional exercises** built on that foundation, not claims that they appeared in the source. One nuance: the lesson calls `add.__get__(None, main_module)` as a demonstration; for normal class attribute lookup, the descriptor's `owner` is a **class**, and a function's `__get__` does not need an artificial module owner.

**Scope:** These are learning implementations. Standard built-ins (`property`, `classmethod`, `staticmethod`, `functools.cached_property`, and normal functions) are preferred for production use when they fit.

In [1]:
import functools
import inspect
import types
import weakref

assert hasattr(lambda x: x, "__get__")
print("Setup OK — standard-library imports and function descriptor check passed.")

Setup OK — standard-library imports and function descriptor check passed.


## Quick reference: the descriptor rules

A descriptor is an object placed on a **class** with `__get__`, `__set__`, or `__delete__`. Normal instance lookup approximately follows: data descriptor on the class/MRO → instance `__dict__` (when present) → non-data descriptor or ordinary class attribute → `__getattr__` fallback. A descriptor defining `__set__` or `__delete__` is *data*, even if its setter just raises. A plain Python function has `__get__` but no setter/deleter and is *non-data*.

For a function `f` stored on a class `C`, `C.f` returns the function, while `obj.f` typically produces a bound method whose `__self__ is obj` and `__func__ is C.f`. Explicit calls to `f.__get__(obj, type(obj))` demonstrate this mechanism. Actual lookup also involves class/metaclass lookup, inheritance, and `super()`, all explored below.

## Problem 01 — Prove what a bound method stores

**Challenge.** Define a class with a method accepting two arguments. Show that class access returns a function; instance access returns a method storing the instance and original function. Call it three different ways without depending on reprs or memory addresses.

**Hint.** Inspect `__self__`, `__func__`, and `types.MethodType`.

### Solution 01 — Prove what a bound method stores

A bound method is a separate callable object carrying `__func__` and `__self__`. It automatically passes `__self__` as the first argument when called. Avoid comparing transient method-object identities as a program invariant.

In [2]:
class P01Vector:
    def __init__(self, x):
        self.x = x

    def shift(self, dx, *, scale=1):
        return self.x + dx * scale

v01 = P01Vector(10)
raw01 = P01Vector.shift
bound01 = v01.shift
explicit01 = raw01.__get__(v01, P01Vector)
manual01 = types.MethodType(raw01, v01)

assert inspect.isfunction(raw01)
assert inspect.ismethod(bound01)
assert bound01.__self__ is v01
assert bound01.__func__ is raw01
assert explicit01.__func__ is manual01.__func__ is raw01
assert explicit01.__self__ is manual01.__self__ is v01
assert v01.shift(3, scale=2) == raw01(v01, 3, scale=2) == manual01(3, scale=2) == 16
print('01 PASS: method = original function + bound instance')

01 PASS: method = original function + bound instance


## Problem 02 — Class access vs. instance access: instrument __get__

**Challenge.** Build a logging non-data descriptor that records both `instance` and `owner`. Use it through a class and two different instances, and verify the owner and the receiver each time.

**Hint.** Class access supplies `instance=None`; instance access receives the actual object.

### Solution 02 — Class access vs. instance access: instrument __get__

The descriptor lives in `vars(P02Person)`. Retrieving it directly from that mapping avoids invoking its `__get__` and is useful when debugging. Never assume an instance access passes `owner=None` during normal attribute lookup.

In [3]:
class P02Observer:
    def __init__(self, func):
        self.func = func
        self.events = []

    def __get__(self, instance, owner=None):
        self.events.append((instance, owner))
        if instance is None:
            return self.func
        return types.MethodType(self.func, instance)

class P02Person:
    def __init__(self, name):
        self.name = name

    def greet(self):  # used only for the implementation function
        return self.name.upper()

    action = P02Observer(greet)

observer02 = vars(P02Person)['action']  # raw descriptor; no __get__ call
p02a, p02b = P02Person('Ada'), P02Person('Grace')
assert P02Person.action is observer02.func
assert p02a.action() == 'ADA'
assert p02b.action() == 'GRACE'
assert observer02.events == [
    (None, P02Person), (p02a, P02Person), (p02b, P02Person)
]
print('02 PASS: correct instance and owner for each lookup')

02 PASS: correct instance and owner for each lookup


## Problem 03 — An instance can shadow an ordinary method

**Challenge.** Override a single object's inherited method by setting an entry in its instance dictionary. Explain why a plain function stored directly on the instance does **not** magically bind `self`. Restore ordinary dispatch without modifying other objects.

**Hint.** Functions are non-data descriptors only when encountered through *class* lookup.

### Solution 03 — An instance can shadow an ordinary method

Non-data descriptors lose lookup precedence to the instance dictionary. The function-binding rule is not applied to a function already retrieved from that dictionary. An explicit `MethodType` can bind it, but removing the override is usually cleaner.

In [4]:
class P03Greeter:
    def greet(self, punctuation='!'):
        return 'original' + punctuation

first03, second03 = P03Greeter(), P03Greeter()
assert first03.greet() == second03.greet() == 'original!'
first03.greet = lambda: 'instance override'
assert first03.greet() == 'instance override'
assert second03.greet() == 'original!'

# Assigning a function into an INSTANCE dict does not invoke function.__get__.
def p03_replacement(self):
    return 'replacement for ' + type(self).__name__

first03.greet = p03_replacement
assert first03.greet is p03_replacement
try:
    first03.greet()
except TypeError:
    pass  # missing required 'self': expected
else:
    raise AssertionError('An instance-stored function was unexpectedly bound')

first03.greet = types.MethodType(p03_replacement, first03)
assert first03.greet() == 'replacement for P03Greeter'
del first03.greet
assert first03.greet() == 'original!'
print('03 PASS: one-object shadowing, explicit binding, restoration')

03 PASS: one-object shadowing, explicit binding, restoration


## Problem 04 — Re-create ordinary function binding with a descriptor

**Challenge.** Write `P04Method` with a stored function and `__get__` so that class access returns the original function and instance access returns a real bound method. Make the descriptor usable as a decorator.

**Hint.** Use `types.MethodType` rather than an ad hoc lambda if you want standard `__func__`/`__self__`.

### Solution 04 — Re-create ordinary function binding with a descriptor

The descriptor stores the function *on the class*, returns that same function for class access, and constructs a bound method for instance access. This intentionally mirrors the supplied lesson, but adds decorator syntax and executable identity tests.

In [5]:
class P04Method:
    def __init__(self, func):
        if not callable(func):
            raise TypeError('P04Method requires a callable')
        self.func = func

    def __get__(self, instance, owner=None):
        return self.func if instance is None else types.MethodType(self.func, instance)

class P04Counter:
    def __init__(self, start=0):
        self.value = start

    @P04Method
    def increment(self, amount=1):
        self.value += amount
        return self.value

counter04 = P04Counter(5)
raw04 = vars(P04Counter)['increment']
assert isinstance(raw04, P04Method)
assert P04Counter.increment is raw04.func
assert counter04.increment(3) == 8
assert counter04.increment.__self__ is counter04
assert counter04.increment.__func__ is raw04.func
assert 'increment' not in vars(counter04)
print('04 PASS: reimplemented normal non-data method binding')

04 PASS: reimplemented normal non-data method binding


## Problem 05 — Why a closure is not quite a method

**Challenge.** Make a second binding descriptor using a nested function and `functools.wraps`. Verify it calls correctly, then demonstrate how its introspection differs from a real bound method. Do not call the two equivalent.

**Hint.** The wrapper closes over an instance, but `inspect.ismethod` tests a particular object type.

### Solution 05 — Why a closure is not quite a method

`functools.wraps` preserves useful metadata (including `__wrapped__`) but cannot turn a closure into a real `method` object. Different introspection is a concrete reason to prefer ordinary functions or `MethodType` unless custom wrapping is needed.

In [6]:
class P05ClosureMethod:
    def __init__(self, func):
        self.func = func

    def __get__(self, instance, owner=None):
        if instance is None:
            return self.func

        @functools.wraps(self.func)
        def call(*args, **kwargs):
            return self.func(instance, *args, **kwargs)
        return call

class P05Example:
    def __init__(self, number):
        self.number = number

    @P05ClosureMethod
    def multiply(self, factor):
        """Return a product."""
        return self.number * factor

p05 = P05Example(7)
wrapped05 = p05.multiply
assert wrapped05(6) == 42
assert inspect.isfunction(wrapped05) and not inspect.ismethod(wrapped05)
assert wrapped05.__name__ == 'multiply'
assert wrapped05.__doc__ == 'Return a product.'
assert wrapped05.__wrapped__ is vars(P05Example)['multiply'].func
assert not hasattr(wrapped05, '__self__')
print('05 PASS: callable closure is not a bound-method object')

05 PASS: callable closure is not a bound-method object


## Problem 06 — Derive descriptor lookup precedence empirically

**Challenge.** Implement one data descriptor and one non-data descriptor, give the instance dictionary conflicting values, and verify that the data descriptor wins but the non-data descriptor is shadowed. Compare direct dictionary reads with ordinary access.

**Hint.** Data descriptors define `__get__` and at least one of `__set__`/`__delete__`.

### Solution 06 — Derive descriptor lookup precedence empirically

Attribute lookup does not simply return the first matching dictionary entry. A data descriptor beats an instance-dictionary entry with the same name; an instance-dictionary entry beats a non-data descriptor.

In [7]:
class P06Data:
    def __get__(self, instance, owner=None):
        return 'data descriptor' if instance is not None else self

    def __set__(self, instance, value):
        instance.__dict__['_data'] = value

class P06NonData:
    def __get__(self, instance, owner=None):
        return 'non-data descriptor' if instance is not None else self

class P06Example:
    data = P06Data()
    nondata = P06NonData()

p06 = P06Example()
p06.__dict__.update({'data': 'instance data', 'nondata': 'instance nondata'})
assert p06.data == 'data descriptor'
assert p06.nondata == 'instance nondata'
assert vars(p06)['data'] == 'instance data'
assert vars(p06)['nondata'] == 'instance nondata'
p06.data = 99
assert p06._data == 99
assert p06.data == 'data descriptor'
del p06.__dict__['nondata']
assert p06.nondata == 'non-data descriptor'
print('06 PASS: data descriptor > instance dict > non-data descriptor')

06 PASS: data descriptor > instance dict > non-data descriptor


## Problem 07 — Build a reusable validated data descriptor

**Challenge.** Create `P07PositiveInt` with `__set_name__`, `__get__`, `__set__`, and `__delete__`. Store the value per instance; reject `bool`, zero, and negative numbers. Expose the descriptor on class access and test missing, set, invalid, and delete cases.

**Hint.** Remember `bool` is a subclass of `int` and a single descriptor object is shared by all instances.

### Solution 07 — Build a reusable validated data descriptor

`__set_name__` gives each descriptor its class-assigned name. The validation runs in `__set__` and the internal storage remains instance-specific. This implementation deliberately requires ordinary instance dictionaries; adapting it for slot-only instances is a separate design decision.

In [8]:
class P07PositiveInt:
    def __set_name__(self, owner, name):
        self.storage_name = '_validated_' + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.storage_name]
        except KeyError:
            raise AttributeError(self.storage_name + ' has not been assigned') from None

    def __set__(self, instance, value):
        if type(value) is not int or value <= 0:
            raise ValueError('value must be a positive int (not bool)')
        instance.__dict__[self.storage_name] = value

    def __delete__(self, instance):
        try:
            del instance.__dict__[self.storage_name]
        except KeyError:
            raise AttributeError('value is not set') from None

class P07Product:
    stock = P07PositiveInt()

item07a, item07b = P07Product(), P07Product()
assert isinstance(P07Product.stock, P07PositiveInt)
for invalid07 in (True, 0, -2, 1.5, '3'):
    try:
        item07a.stock = invalid07
    except ValueError:
        pass
    else:
        raise AssertionError(f'Accepted bad stock: {invalid07!r}')
item07a.stock, item07b.stock = 3, 7
assert (item07a.stock, item07b.stock) == (3, 7)
assert 'stock' not in vars(item07a)
assert vars(item07a)['_validated_stock'] == 3
del item07a.stock
try:
    _ = item07a.stock
except AttributeError:
    pass
else:
    raise AssertionError('Deleted value should be missing')
assert item07b.stock == 7
print('07 PASS: validation, independent storage, deletion')

07 PASS: validation, independent storage, deletion


## Problem 08 — A read-only descriptor still takes priority

**Challenge.** Construct a read-only **data** descriptor that raises in `__set__`. Insert a same-named value into `__dict__` manually and prove normal lookup still calls the descriptor. Contrast a bare `@property`.

**Hint.** A setter that raises is still a setter for precedence purposes.

### Solution 08 — A read-only descriptor still takes priority

Read-only means assignments fail through normal syntax; it does **not** make Python a security boundary against deliberate modification of internals. A property without a user-defined setter is still a data descriptor.

In [9]:
class P08ReadOnly:
    def __get__(self, instance, owner=None):
        return self if instance is None else 'protected'

    def __set__(self, instance, value):
        raise AttributeError('read-only field')

class P08Account:
    token = P08ReadOnly()

    @property
    def status(self):
        return 'active'

p08 = P08Account()
vars(p08)['token'] = 'spoofed'
vars(p08)['status'] = 'spoofed'
assert p08.token == 'protected'
assert p08.status == 'active'
for field08 in ('token', 'status'):
    try:
        setattr(p08, field08, 'new')
    except AttributeError:
        pass
    else:
        raise AssertionError(f'{field08} should reject assignment')
print('08 PASS: raising setters and properties are data descriptors')

08 PASS: raising setters and properties are data descriptors


## Problem 09 — Method lookup on instances without __dict__

**Challenge.** Use `__slots__` to prevent arbitrary per-instance attributes, while keeping normal method binding working. Inspect where the slot descriptor lives and test failed method shadowing.

**Hint.** Class-body functions bind without storing bound method objects in `obj.__dict__`.

### Solution 09 — Method lookup on instances without __dict__

Slots create class-level slot descriptors and (here) omit the instance dictionary. Function binding works independently of the presence of `__dict__`; the missing dict only prevents ordinary instance shadowing.

In [10]:
class P09Slotted:
    __slots__ = ('name',)

    def __init__(self, name):
        self.name = name

    def label(self):
        return '[' + self.name + ']'

p09 = P09Slotted('test')
assert not hasattr(p09, '__dict__')
assert p09.label() == '[test]'
assert p09.label.__self__ is p09
assert P09Slotted.__dict__['name'].__get__(p09, P09Slotted) == 'test'
try:
    p09.label = lambda: 'override'
except AttributeError:
    pass
else:
    raise AssertionError('A slot-only object accepted an undeclared attribute')
assert p09.label() == '[test]'
print('09 PASS: methods bind on slot-only instances')

09 PASS: methods bind on slot-only instances


## Problem 10 — Compare method, classmethod and staticmethod

**Challenge.** Create all three function-like attributes in one class. Check the receiver bound in each case when accessed from both the base class and a subclass. Inspect their raw descriptor objects without triggering lookup.

**Hint.** `classmethod` binds a class, `staticmethod` binds nothing, and a regular method binds an instance.

### Solution 10 — Compare method, classmethod and staticmethod

A `classmethod` receives the class through which it is accessed (including subclasses); a `staticmethod` returns the wrapped callable without injecting a receiver. Use `inspect.getattr_static` when you need the *raw* descriptor rather than the result of calling its `__get__`.

In [11]:
class P10Base:
    def regular(self):
        return type(self).__name__

    @classmethod
    def class_kind(cls):
        return cls.__name__

    @staticmethod
    def calculate(x, y):
        return x + y

class P10Child(P10Base):
    pass

p10 = P10Child()
assert p10.regular() == 'P10Child'
assert p10.regular.__self__ is p10
assert P10Child.class_kind() == p10.class_kind() == 'P10Child'
assert P10Child.class_kind.__self__ is P10Child
assert p10.calculate(2, 5) == P10Child.calculate(2, 5) == 7
assert inspect.isfunction(P10Child.calculate)
assert isinstance(inspect.getattr_static(P10Base, 'class_kind'), classmethod)
assert isinstance(inspect.getattr_static(P10Base, 'calculate'), staticmethod)
assert inspect.getattr_static(P10Base, 'regular') is P10Base.regular
print('10 PASS: distinct binding contracts and subclass dispatch')

10 PASS: distinct binding contracts and subclass dispatch


## Problem 11 — Decorator order changes the object being wrapped

**Challenge.** Create a tracing decorator using `functools.wraps`. Combine it with `classmethod` and `staticmethod` using reliable ordering, verify metadata and subclass binding, and show what object the inner decorator saw.

**Hint.** Place `@classmethod` or `@staticmethod` above `@trace` when `trace` expects a function.

### Solution 11 — Decorator order changes the object being wrapped

Decorators apply bottom-up: `@classmethod` wraps the **result** of `@p11_trace`. Reversing them would hand a descriptor, not a plain function, to this particular `trace` implementation; don't assert all decorators are universally interchangeable.

In [12]:
p11_events = []

def p11_trace(func):
    assert inspect.isfunction(func), 'p11_trace decorates a plain function'

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        p11_events.append((func.__name__, len(args)))
        return func(*args, **kwargs)
    return wrapper

class P11Base:
    @classmethod
    @p11_trace
    def identify(cls, suffix=''):
        """Describe the class."""
        return cls.__name__ + suffix

    @staticmethod
    @p11_trace
    def twice(number):
        """Double a number."""
        return 2 * number

class P11Child(P11Base):
    pass

assert P11Child.identify('?') == 'P11Child?'
assert P11Child.twice(6) == 12
assert P11Child.identify.__self__ is P11Child
assert P11Child.identify.__name__ == 'identify'
assert P11Child.identify.__doc__ == 'Describe the class.'
assert p11_events == [('identify', 2), ('twice', 1)]
raw11 = inspect.getattr_static(P11Base, 'identify')
assert isinstance(raw11, classmethod)
assert raw11.__func__.__wrapped__.__name__ == 'identify'
print('11 PASS: decorator composition, metadata and receiver counts')

11 PASS: decorator composition, metadata and receiver counts


## Problem 12 — Observe owner and instance in inherited descriptor lookup

**Challenge.** Put a logging descriptor on a base class and read it through the base, subclass, and subclass instance. Establish whether normal inherited access supplies the declaring class or the accessing class as `owner`.

**Hint.** Read the descriptor with `vars(Base)` to avoid triggering its log.

### Solution 12 — Observe owner and instance in inherited descriptor lookup

The descriptor's storage location does not automatically determine `owner`. For ordinary class or instance lookup shown here, `owner` is the class used for attribute access / the instance's type. `super()` has specialized behavior, studied next.

In [13]:
class P12Recorder:
    def __init__(self):
        self.calls = []

    def __get__(self, instance, owner=None):
        self.calls.append((instance, owner))
        return (instance, owner)

class P12Base:
    signal = P12Recorder()

class P12Child(P12Base):
    pass

raw12 = vars(P12Base)['signal']
assert P12Base.signal == (None, P12Base)
assert P12Child.signal == (None, P12Child)
child12 = P12Child()
assert child12.signal == (child12, P12Child)
assert raw12.calls == [
    (None, P12Base), (None, P12Child), (child12, P12Child)
]
print('12 PASS: owner on ordinary inherited access is the accessing class')

12 PASS: owner on ordinary inherited access is the accessing class


## Problem 13 — Use super() without losing binding

**Challenge.** Override a method in a subclass, use `super()` to call its base implementation, and show how the base implementation can still operate on the subclass instance. Verify `super(Sub, obj).method` carries the original instance as `__self__`.

**Hint.** `super()` changes which class in the MRO supplies the attribute; it does not replace the receiver.

### Solution 13 — Use super() without losing binding

`super()` performs a special MRO search beginning after a specified class. The selected base function is still invoked with the original subclass instance, so subclass state can be used in the base implementation.

In [14]:
class P13Base:
    def render(self):
        return f'base:{self.value}'

class P13Child(P13Base):
    def __init__(self, value):
        self.value = value

    def render(self):
        return super().render() + ':child'

p13 = P13Child(9)
base_method13 = super(P13Child, p13).render
assert p13.render() == 'base:9:child'
assert base_method13() == 'base:9'
assert base_method13.__self__ is p13
assert base_method13.__func__ is P13Base.render
assert P13Base.render(p13) == 'base:9'
print('13 PASS: super selects a base function and binds the same receiver')

13 PASS: super selects a base function and binds the same receiver


## Problem 14 — Inspect raw attributes without triggering descriptors

**Challenge.** Build a descriptor with an observable side effect. Compare `getattr`, `vars(cls)`, and `inspect.getattr_static` on a class and instance. Establish which inspection forms leave the descriptor untouched.

**Hint.** Static inspection returns the descriptor object rather than evaluating `__get__`.

### Solution 14 — Inspect raw attributes without triggering descriptors

This is important in debugging and tooling: property getters and arbitrary descriptors can raise, mutate state, perform I/O, or be expensive. `inspect.getattr_static` avoids dynamic attribute lookup, although the raw object may require careful interpretation.

In [15]:
class P14SideEffect:
    def __init__(self):
        self.count = 0

    def __get__(self, instance, owner=None):
        self.count += 1
        return 'evaluated'

class P14Example:
    value = P14SideEffect()

raw14 = vars(P14Example)['value']
p14 = P14Example()
assert inspect.getattr_static(P14Example, 'value') is raw14
assert inspect.getattr_static(p14, 'value') is raw14
assert raw14.count == 0
assert getattr(p14, 'value') == 'evaluated'
assert P14Example.value == 'evaluated'
assert raw14.count == 2
print('14 PASS: static inspection avoids descriptor execution')

14 PASS: static inspection avoids descriptor execution


## Problem 15 — Override __getattribute__ safely

**Challenge.** Implement a class that logs attribute names but otherwise preserves Python's descriptor machinery. Demonstrate a property is still invoked exactly once and a missing attribute can fall through to `__getattr__`.

**Hint.** Call `object.__getattribute__(self, name)` rather than `self.__dict__` inside the override.

### Solution 15 — Override __getattribute__ safely

Overriding `__getattribute__` without delegating to `object.__getattribute__` can silently bypass descriptors or trigger infinite recursion. `__getattr__` is a fallback after an `AttributeError`, not a substitute for the main lookup protocol.

In [16]:
class P15Observed:
    def __init__(self):
        object.__setattr__(self, 'reads', [])
        object.__setattr__(self, 'property_calls', 0)

    def __getattribute__(self, name):
        if name not in ('reads', 'property_calls'):
            object.__getattribute__(self, 'reads').append(name)
        return object.__getattribute__(self, name)

    @property
    def score(self):
        current = object.__getattribute__(self, 'property_calls')
        object.__setattr__(self, 'property_calls', current + 1)
        return 100

    def __getattr__(self, name):
        if name == 'fallback':
            return 'from __getattr__'
        raise AttributeError(name)

p15 = P15Observed()
assert p15.score == 100
assert p15.property_calls == 1
assert p15.fallback == 'from __getattr__'
assert p15.reads == ['score', 'fallback']
try:
    p15.unknown
except AttributeError:
    pass
else:
    raise AssertionError('Unexpected attribute exists')
assert p15.reads[-1] == 'unknown'
print('15 PASS: delegation preserves property and fallback behavior')

15 PASS: delegation preserves property and fallback behavior


## Problem 16 — Explore functools.partialmethod binding

**Challenge.** Make one method with two arguments; use `functools.partialmethod` in the class to pre-fill one. Compare it to a `functools.partial` created at runtime from a bound method and ensure both respect the instance.

**Hint.** `partialmethod` implements descriptor-aware behavior, unlike a plain partial object used blindly as a class attribute.

### Solution 16 — Explore functools.partialmethod binding

`partialmethod` is meant for methods declared on classes. A regular `partial` is excellent when starting from an *already bound* callable, but does not itself model class-method descriptor binding in the same way.

In [17]:
class P16Formatter:
    def __init__(self, prefix):
        self.prefix = prefix

    def render(self, value, *, end='.'):
        return f'{self.prefix}:{value}{end}'

    render_warning = functools.partialmethod(render, 'WARNING', end='!')

p16a, p16b = P16Formatter('A'), P16Formatter('B')
assert p16a.render_warning() == 'A:WARNING!'
assert p16b.render_warning() == 'B:WARNING!'
assert p16a.render_warning(end='?') == 'A:WARNING?'
ordinary_partial16 = functools.partial(p16a.render, 'WARNING', end='!')
assert ordinary_partial16() == p16a.render_warning()
assert isinstance(inspect.getattr_static(P16Formatter, 'render_warning'), functools.partialmethod)
assert ordinary_partial16.func.__self__ is p16a
print('16 PASS: partialmethod preserves per-instance binding')

16 PASS: partialmethod preserves per-instance binding


## Problem 17 — Design a non-data cached computation with invalidation

**Challenge.** Create a descriptor that computes a value once per instance, caches it in `__dict__` under the public attribute name, and exposes an explicit invalidation operation. Prove different instances do not share values and that cache invalidation recomputes.

**Hint.** Non-data descriptors are shadowed by the instance dict: use that behavior intentionally.

### Solution 17 — Design a non-data cached computation with invalidation

This exploits non-data precedence, much like `functools.cached_property`. Document cache invalidation! It uses `instance.__dict__`, so it is not directly compatible with slot-only classes without a dict. For ordinary production code, prefer the built-in utility when it meets your needs.

In [18]:
class P17CachedValue:
    def __init__(self, func):
        self.func = func
        functools.update_wrapper(self, func)

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        value = self.func(instance)
        instance.__dict__[self.name] = value
        return value

    def invalidate(self, instance):
        instance.__dict__.pop(self.name, None)

class P17Report:
    def __init__(self, data):
        self.data = list(data)
        self.calls = 0

    @P17CachedValue
    def total(self):
        """Compute the total once until invalidated."""
        self.calls += 1
        return sum(self.data)

r17a, r17b = P17Report([1, 2]), P17Report([10])
assert r17a.total == r17a.total == 3
assert r17a.calls == 1
assert r17b.total == 10 and r17b.calls == 1
r17a.data.append(5)
assert r17a.total == 3  # cached value intentionally becomes stale
P17Report.total.invalidate(r17a)
assert r17a.total == 8 and r17a.calls == 2
assert r17b.calls == 1
assert P17Report.total.__name__ == 'total'
print('17 PASS: per-instance cache, staleness and explicit invalidation')

17 PASS: per-instance cache, staleness and explicit invalidation


## Problem 18 — Count method access per instance without retaining instances

**Challenge.** Implement a non-data descriptor that records how many times each object accesses a method using `weakref.WeakKeyDictionary`, but returns a fresh bound method without caching it. Prove access counts are independent, and explain the weak-reference constraint.

**Hint.** Store only integer counts in the weak map; a bound method stored as a *value* would retain its instance.

### Solution 18 — Count method access per instance without retaining instances

`WeakKeyDictionary` requires weak-referenceable, hashable instance keys. This exercise stores integers as values: caching `types.MethodType(func, instance)` as a value in a weak-key dictionary can *keep the instance alive* via the method, defeating weak-key cleanup.

In [19]:
class P18CountAccess:
    def __init__(self, func):
        self.func = func
        self.counts = weakref.WeakKeyDictionary()

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        self.counts[instance] = self.counts.get(instance, 0) + 1
        return types.MethodType(self.func, instance)

    def count_for(self, instance):
        return self.counts.get(instance, 0)

class P18Worker:
    @P18CountAccess
    def work(self, number):
        return number ** 2

w18a, w18b = P18Worker(), P18Worker()
descriptor18 = P18Worker.work
assert descriptor18.count_for(w18a) == 0
assert w18a.work(3) == 9
assert w18a.work(4) == 16
assert w18b.work(5) == 25
assert descriptor18.count_for(w18a) == 2
assert descriptor18.count_for(w18b) == 1

# Weak-reference support is opt-in for some slot-only classes.
class P18NotWeakrefable:
    __slots__ = ()

try:
    weakref.ref(P18NotWeakrefable())
except TypeError:
    pass
else:
    raise AssertionError('Expected this slot-only class to reject weak references')
print('18 PASS: counters avoid storing instance-retaining bound methods')

18 PASS: counters avoid storing instance-retaining bound methods


## Problem 19 — Understand __set_name__ lifecycle and dynamic assignment

**Challenge.** Show that a descriptor receives its name automatically during class creation, but does not receive `__set_name__` automatically when assigned to a class afterward. Initialize a dynamically added descriptor explicitly and verify both cases.

**Hint.** Use a descriptor recording `(owner, name)` pairs and `vars(cls)` to read it raw.

### Solution 19 — Understand __set_name__ lifecycle and dynamic assignment

`type.__new__` sends `__set_name__` notifications while creating a class. Ordinary `Class.attr = descriptor` assignment is not class creation; call the hook yourself if the descriptor needs initialization. Avoid reusing one name-storing descriptor under multiple names unless designed for it.

In [20]:
class P19Named:
    def __init__(self):
        self.owner = None
        self.name = None
        self.notifications = 0

    def __set_name__(self, owner, name):
        self.owner, self.name = owner, name
        self.notifications += 1

    def __get__(self, instance, owner=None):
        return self if instance is None else self.name

class P19Example:
    built_in_class_body = P19Named()

body19 = vars(P19Example)['built_in_class_body']
assert (body19.owner, body19.name, body19.notifications) == (
    P19Example, 'built_in_class_body', 1
)
dynamic19 = P19Named()
P19Example.added_later = dynamic19
assert dynamic19.notifications == 0
assert P19Example().added_later is None
dynamic19.__set_name__(P19Example, 'added_later')
assert P19Example().added_later == 'added_later'
assert dynamic19.notifications == 1
print('19 PASS: class creation calls __set_name__; later assignment does not')

19 PASS: class creation calls __set_name__; later assignment does not


## Problem 20 — Special-method lookup bypasses instance shadowing

**Challenge.** Give an object a normal method and `__len__`, then shadow both in its instance dictionary. Show why explicit `obj.__len__()` and implicit `len(obj)` can disagree. Remove the overrides and verify restoration.

**Hint.** Language operations such as `len(obj)` look for special methods on the type, not via ordinary instance access.

### Solution 20 — Special-method lookup bypasses instance shadowing

`obj.__len__` uses ordinary lookup and sees the instance override. `len(obj)` performs special-method lookup on the type, deliberately bypassing the instance dictionary (and in some cases custom instance lookup hooks).

In [21]:
class P20Container:
    def __init__(self, items):
        self.items = list(items)

    def size(self):
        return len(self.items)

    def __len__(self):
        return len(self.items)

p20 = P20Container([1, 2, 3])
assert p20.size() == len(p20) == 3
p20.size = lambda: 77
p20.__len__ = lambda: 88
assert p20.size() == 77
assert p20.__len__() == 88
assert len(p20) == 3  # special-method lookup goes through the TYPE
del p20.size
del p20.__len__
assert p20.size() == p20.__len__() == len(p20) == 3
print('20 PASS: special-method lookup differs from ordinary attribute access')

20 PASS: special-method lookup differs from ordinary attribute access


## Problem 21 — A callable object is not automatically a bound method

**Challenge.** Store a callable instance on a class and verify instance access returns that same callable without binding. Then add `__get__` to a different callable class and demonstrate real per-instance binding.

**Hint.** Having `__call__` is not the same as implementing the descriptor protocol.

### Solution 21 — A callable object is not automatically a bound method

A generic callable stored in a class is returned unchanged unless it also implements `__get__`. A descriptor may return a bound method even if the underlying callable is an object rather than a Python function.

In [22]:
class P21Callable:
    def __call__(self, value):
        return value * 2

plain_callable21 = P21Callable()

class P21First:
    action = plain_callable21

f21 = P21First()
assert f21.action is plain_callable21
assert f21.action(7) == 14
assert not inspect.ismethod(f21.action)

class P21BindingCallable:
    def __call__(self, instance, value):
        return instance.factor * value

    def __get__(self, instance, owner=None):
        return self if instance is None else types.MethodType(self, instance)

class P21Second:
    action = P21BindingCallable()

    def __init__(self, factor):
        self.factor = factor

s21 = P21Second(5)
assert s21.action(7) == 35
assert s21.action.__self__ is s21
assert s21.action.__func__ is P21Second.action
print('21 PASS: __call__ makes callable; __get__ controls binding')

21 PASS: __call__ makes callable; __get__ controls binding


## Problem 22 — Capstone: a validated, auditable method descriptor

**Challenge.** Implement `@P22CheckedMethod` for a `withdraw(amount, *, note='')` method. Validate that `amount` is a positive **integer (not bool)** whether passed positionally or by keyword; log only successful calls, preserving method metadata and subclass binding. Keep the account's insufficient-funds rule in the business method. Test both class- and instance-style invocation, invalid inputs, balance integrity, and metadata.

**Hint.** Use `inspect.signature(func).bind(instance, *args, **kwargs)`, `apply_defaults`, and `functools.wraps`. Binding validates argument shape and lets you read `amount` consistently.

### Solution 22 — Capstone: a validated, auditable method descriptor

The descriptor returns a real bound method for instances, and the wrapped function for class access. Signature binding handles positional/keyword variants and raises on missing/duplicate arguments. Validation happens before the business logic; the audit occurs only after successful completion. This is an educational demonstration, not a security control or a substitute for transaction-safe persistence.

In [23]:
class P22CheckedMethod:
    def __init__(self, func):
        self.func = func
        self.signature = inspect.signature(func)
        self.successes = []

        @functools.wraps(func)
        def checked(receiver, *args, **kwargs):
            # Bind before executing the business method: errors do not mutate state.
            arguments = self.signature.bind(receiver, *args, **kwargs)
            arguments.apply_defaults()
            amount = arguments.arguments['amount']
            if type(amount) is not int or amount <= 0:
                raise ValueError('amount must be a positive int (not bool)')
            result = self.func(receiver, *args, **kwargs)
            # Only successful completions are audited; no user secrets logged.
            self.successes.append((type(receiver).__name__, amount))
            return result

        self.checked = checked

    def __get__(self, instance, owner=None):
        if instance is None:
            return self.checked
        return types.MethodType(self.checked, instance)

class P22Account:
    def __init__(self, balance):
        self.balance = balance

    @P22CheckedMethod
    def withdraw(self, amount, *, note=''):
        """Remove an authorized, positive integer amount."""
        if amount > self.balance:
            raise ValueError('insufficient funds')
        self.balance -= amount
        return self.balance, note

class P22Savings(P22Account):
    pass

account22 = P22Savings(100)
descriptor22 = vars(P22Account)['withdraw']
assert isinstance(descriptor22, P22CheckedMethod)
assert account22.withdraw(20, note='groceries') == (80, 'groceries')
assert P22Savings.withdraw(account22, amount=5) == (75, '')
assert descriptor22.successes == [('P22Savings', 20), ('P22Savings', 5)]
assert account22.withdraw.__self__ is account22
assert account22.withdraw.__func__ is descriptor22.checked
assert account22.withdraw.__func__.__wrapped__ is descriptor22.func
assert account22.withdraw.__name__ == 'withdraw'
assert account22.withdraw.__doc__ == 'Remove an authorized, positive integer amount.'

for bad22 in (True, 0, -1, 1.5, '3'):
    try:
        account22.withdraw(amount=bad22)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Accepted invalid amount: {bad22!r}')

try:
    account22.withdraw(1000)
except ValueError as exc:
    assert str(exc) == 'insufficient funds'
else:
    raise AssertionError('Accepted an overdraft')

try:
    account22.withdraw()  # argument shape checked by inspect.Signature.bind
except TypeError:
    pass
else:
    raise AssertionError('Accepted a missing amount')

assert account22.balance == 75
assert len(descriptor22.successes) == 2
print('22 PASS: validated calls, subclass receiver, clean audit, preserved metadata')

22 PASS: validated calls, subclass receiver, clean audit, preserved metadata


## Further challenge set (with answers)

These short reasoning questions test generalization without repeating the full implementations:

1. **Q:** Why does `obj.method` usually bind but a function assigned to `obj.method` not bind? **A:** The first function is reached through class descriptor lookup; the second is retrieved directly from the instance dictionary.
2. **Q:** How do you inspect the descriptor itself when `Class.attr` calls `__get__`? **A:** Use `vars(Class)['attr']` for a directly declared attribute or `inspect.getattr_static` for inherited/static lookup.
3. **Q:** Why does `obj.read_only` ignore `obj.__dict__['read_only']`? **A:** A read-only descriptor that implements a raising `__set__` is still a data descriptor.
4. **Q:** Which object does `obj.method.__self__` refer to? **A:** The instance for a regular instance method; the bound class for a class method.
5. **Q:** Why might caching bound methods in a `WeakKeyDictionary` still leak instances? **A:** The cached bound-method value holds a strong reference back to the key instance.
6. **Q:** Why can `len(obj)` disagree with `obj.__len__()`? **A:** Implicit special-method lookup happens on the type rather than ordinary instance lookup.
7. **Q:** How can a non-data descriptor cache a computed value? **A:** Write the result to the instance dict under the same public attribute name; later lookup sees the cached entry first.
8. **Q:** Does dynamically assigning a descriptor call `__set_name__`? **A:** No; automatic name notifications happen at class creation, not on later assignment.

**Caution:** `inspect.getattr_static` and the simplified precedence summary describe ordinary object lookup; metaclasses, `super()`, custom `__getattribute__`, and built-in special-method lookup can add important details.

In [24]:
# Final cross-problem smoke test: each named descriptor is present and callable as designed.
assert P04Counter.increment(P04Counter(1), 2) == 3
assert isinstance(P07Product.stock, P07PositiveInt)
assert isinstance(P17Report.total, P17CachedValue)
assert isinstance(vars(P22Account)['withdraw'], P22CheckedMethod)
print('ALL 22 WORKED PROBLEMS AND FINAL CROSS-CHECK COMPLETED SUCCESSFULLY')

ALL 22 WORKED PROBLEMS AND FINAL CROSS-CHECK COMPLETED SUCCESSFULLY
